# Building goSPL inputs: global erosion–deposition over 1 Myr

This notebook prepares the **input files** for a global `gospl` landscape evolution run that simulates erosion and deposition over a 1 Myr interval starting at 251 Ma. It (1) generates an unstructured spherical mesh with JIGSAW/MPAS at a chosen resolution, (2) interpolates regular lat/lon forcing grids (paleo-elevation, precipitation, effective elastic thickness) onto that mesh, and (3) writes the compressed `.npz` mesh and forcing files that `gospl` reads.

By the end you will have an `input_<width>` mesh directory and a `vars_<width>` folder containing `mesh.npz` (coordinates, connectivity, elevation) and `forcing251.npz` (precipitation and tectonic forcing). These drive the stream-power erosion $E = K A^m S^n$ and hillslope diffusion $\partial z/\partial t = \kappa \nabla^2 z + U$ solved by `gospl`.

**Tools used:** [UXarray](https://uxarray.readthedocs.io), [JIGSAW](https://github.com/dengwirda/jigsaw), [MPAS-Tools](https://github.com/MPAS-Dev/MPAS-Tools), `xarray`.

Import required Python packages for this notebook.


In [ ]:
import importlib, subprocess, sys

def ensure_installed(package):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

ensure_installed("pysheds")

## Imports

Standard scientific-Python stack plus the project helper module `scripts.umeshFcts` (aliased `ufcts`), which wraps the mesh-building and grid-interpolation routines used below. `xarray` handles the netCDF forcing grids; `numpy` writes the final compressed arrays. The `pysheds` dependency is installed in the cell above (it is used elsewhere in the workflow for drainage analysis).

Import required Python packages for this notebook.


In [ ]:
import os
import shutil
import numpy as np
from scipy import constants
import xarray as xr

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from scripts import umeshFcts as ufcts

# Create a global mesh for goSPL

Create an unstructured grid for a given cell width. The method relies on the UXarray and jigsaw libraries.

**In case where the mesh already exists it will not be recreated.**

Spherical mesh resolution km

| cell_width | edge_min  | edge_max | edge_mean | nodeNb |
| ---------- | ----------  | ---------- | ---------- | ---------- |
| 5 | 1.1 | 4.5 | 2.8 | 23632811 |
| 8 |  1.8 | 7.2 | 4.6  | 9236387 | 
| 10 | 2.2 | 8.9 | 5.7 | 5912778 |
| 15 | 3.3 | 13.1 | 8.6 | 2629742 |
| 20 | 4.5 | 18 | 11.5 | 1480168 |
| 25 | 5.6 | 22.4 | 14.4 | 947701 |
| 30 | 6.8 | 26.4 | 17.2 | 658525 |
| 35 | 8 | 30.5 | 20.1 |  484009 |

In [ ]:
widthCell = 40
input_path = "input_"+str(widthCell) 

# Build the mesh
ufcts.buildGlobalMeshSimple(widthCell, input_path)

Run shell commands for file or mesh management.


The source grid `data/251Ma.nc` is a regular 0.25° lat/lon dataset (721 × 1441) for the 251 Ma time step. Its variables are:

| Variable | Symbol | Units | Physical meaning |
| -------- | ------ | ----- | ---------------- |
| `h` | $z$ | m | Paleo-elevation / bathymetry (initial surface) |
| `rain` | $P$ | m/yr | Precipitation rate forcing runoff and discharge |
| `te` | $T_e$ | m | Effective elastic thickness of the lithosphere (flexure) |

In [ ]:
!mv mesh.jig mesh.log mesh.msh mesh-MESH.msh mesh-HFUN.msh mesh_triangles.nc cellWidthVsLatLon.nc $input_path

## Map variables on the UGRID 

We will now map global variables on this unstructured grid. In goSPL, typical variables would be:

- elevation (in m)
- vertical and horizontal tectonic forcing (displacement rates in m/yr)
- precipitation (in m/yr)
- dynamic topography (in m/yr)

Usually they will be provided in the form of `netcdf` or `geotiff` files. In both cases, the `xarray` or `rioxarray` libraries will allow you to open those files conveniently.

> Here we will use a netcdf grid containing all of these variables (except dynamic topography) for a give time interval.

In [ ]:
# Loading the nc regular file
ncgrid = xr.open_dataset('data/251Ma.nc')
ncgrid

In case the file contains more variables than the ones you need for goSPL, you can select only the necessary ones:

In [ ]:
# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
mapds = xr.open_dataset(ufile) 

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(widthCell)
var_name = 'step_251'
if os.path.exists(var_path):
    shutil.rmtree(var_path)
ufcts.inter2UGRID(ncgrid,mapds,var_path,var_name,type='face')
data_ds = xr.open_dataset(var_path + '/' + var_name + '.nc')

In the `var_path` folder, you will find interpolated variables for the the UGRID (one file per variable) 

## Extract mesh geometry

We pull the Cartesian node coordinates (`xCell`, `yCell`, `zCell`, in metres on the sphere) and the triangular connectivity (`cellsOnVertex`, converted to 0-based indexing) that `gospl` needs. The Voronoi `dcEdge` distances give the spacing statistics for this resolution; here the mean edge length is about 40 km, consistent with the requested `widthCell`.

In [ ]:
# Extract nodes and faces information
n_nodes = mapds.dims['nCells']
ucoords = np.zeros((n_nodes, 3))
ucoords[:, 0] = mapds['xCell'].values
ucoords[:, 1] = mapds['yCell'].values
ucoords[:, 2] = mapds['zCell'].values
ufaces = mapds['cellsOnVertex'].values - 1 
print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

# Get information about your mesh:
dcEdge = mapds['dcEdge'].values  # in metres
edge_min = np.round(dcEdge.min() /1000.+0.,2)
edge_max = np.round(dcEdge.max() /1000.+0.,2)
edge_mean = np.round(dcEdge.mean() /1000.+0.,2)
print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

Save voronoi mesh for visualisation purposes


In [ ]:
# Save voronoi mesh for visualisation purposes
saveVoro = False

if saveVoro:
    from mpas_tools.viz.paraview_extractor import extract_vtk
    extract_vtk(
            filename_pattern=ufile,
            variable_list='areaCell',
            dimension_list=['maxEdges=','nVertLevels=', 'nParticles='], 
            mesh_filename=ufile,
            out_dir=input_path, 
            ignore_time=True,
            # lonlat=True,
            xtime='none'
        )
    print("You could now visualise in Paraview (wireframe) the produced voronoi mesh!")
    print("This is a vtp mesh called: ", input_path+'/staticFieldsOnCells.vtp')

## (Optional) Export the Voronoi mesh for inspection

Set `saveVoro = True` to export the dual Voronoi cells (`areaCell`) as a `.vtp` file you can open in ParaView as a wireframe. This is a sanity check on mesh quality and resolution, and is skipped by default to save time.

> You might want to check that everything went according to plan and look at the mesh and variables that will be used in goSPL.

To do so, we will build a `vtk` file that could be visualised in Paraview...

In [ ]:
checkMesh = False

if checkMesh:
    import meshio

    paleovtk = input_path+"/init.vtk"

    vlist = list(data_ds.keys())
    vdata = []
    for k in vlist:
        vdata.append(data_ds[k].values)

    list_data = dict.fromkeys(el for el in vlist)
    list_data.update((k, vdata[i]) for i, k in enumerate(list_data))

    # Define mesh
    vis_mesh = meshio.Mesh(ucoords, {"triangle": ufaces}, 
                           point_data = list_data,
                        )
    # Write it disk
    meshio.write(paleovtk, vis_mesh)
    print("Writing VTK input file as {}".format(paleovtk))

Set up file paths and names for mesh and output files.


## Write the goSPL mesh file

Save the final mesh as a compressed `mesh.npz` in the `vars_<width>` folder. `gospl` reads three arrays from it:

- `v` — node coordinates (`ucoords`, Cartesian, metres)
- `c` — triangle connectivity (`ufaces`)
- `z` — initial elevation field (`data_ds.h`, metres)

In [ ]:
meshname = var_path+"/mesh"
np.savez_compressed(meshname, v=ucoords, c=ufaces, 
                    z=data_ds.h.data
                    )

Now we save the forcing conditions (displacement rates, tectonic, precipitation...). Here you have the option to also add the next time step elevation, this will then be used in goSPL to force the model to match with the next paleo-elevation for specific regions (by defining the `zfit` parameter in the input file).

In [ ]:
forcname = var_path+"/forcing251"

np.savez_compressed(forcname, 
                    te=data_ds.te.data, 
                    r=data_ds.rain.data,
                    )


This cell is currently empty.
